# 🏥 Healthcare Chatbot for Symptom Analysis

**Author:** Namitha Padigapati  
**Tools:** Python, spaCy, Scikit-learn (SVM, Decision Tree), Speech-to-Text API  
**Goal:** Build an NLP-powered symptom analysis chatbot that predicts conditions and triage levels, supporting both text and voice input

---

## 📌 Problem Statement
Patients often struggle to assess symptom urgency. This chatbot uses SVM and Decision Tree classifiers with spaCy NLP pipelines to predict likely conditions and triage levels (Emergency / Urgent / Non-Urgent), helping users decide whether to seek immediate care.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
print('Libraries loaded successfully ✅')
print('Note: spaCy used in production chatbot pipeline (see chatbot.py)')

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('symptom_condition_pairs.csv')
print(f'Dataset Shape: {df.shape}')
print(f'Unique Conditions: {df["condition"].nunique()}')
print(f'Triage Distribution:\n{df["triage_level"].value_counts()}')
df.head(10)

In [ ]:
print('=== Condition Distribution ===')
print(df['condition'].value_counts())
print('\n=== Severity Distribution ===')
print(df['severity'].value_counts())
print('\n=== Age Group Distribution ===')
print(df['age_group'].value_counts())

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor='#0a0a0f')
fig.suptitle('Healthcare Chatbot — EDA', color='#00e676', fontsize=14, fontweight='bold')

# Triage distribution
axes[0,0].set_facecolor('#12121a')
triage_counts = df['triage_level'].value_counts()
colors = ['#ff6b6b','#ff9800','#00e676']
wedges, texts, autotexts = axes[0,0].pie(
    triage_counts.values, labels=triage_counts.index, colors=colors,
    autopct='%1.1f%%', startangle=90,
    textprops={'color':'#e8e8f0'},
    wedgeprops={'edgecolor':'#0a0a0f','linewidth':2})
for at in autotexts: at.set_color('#0a0a0f'); at.set_fontweight('bold')
axes[0,0].set_title('Triage Level Distribution', color='#e8e8f0', fontsize=11)

# Top conditions
axes[0,1].set_facecolor('#12121a')
top_cond = df['condition'].value_counts().head(8).sort_values()
axes[0,1].barh(top_cond.index, top_cond.values,
               color=['#00e5ff' if v==top_cond.max() else '#7c4dff' for v in top_cond.values],
               edgecolor='#0a0a0f')
axes[0,1].set_title('Top Conditions by Frequency', color='#e8e8f0', fontsize=11)
axes[0,1].tick_params(colors='#8888aa')
axes[0,1].spines['top'].set_visible(False); axes[0,1].spines['right'].set_visible(False)

# Severity by triage
axes[1,0].set_facecolor('#12121a')
sev_triage = pd.crosstab(df['severity'], df['triage_level'])
sev_triage.plot(kind='bar', ax=axes[1,0], color=['#ff6b6b','#00e676','#ff9800'],
                edgecolor='#0a0a0f', width=0.7)
axes[1,0].set_title('Severity by Triage Level', color='#e8e8f0', fontsize=11)
axes[1,0].tick_params(colors='#8888aa', axis='both', rotation=0)
axes[1,0].legend(fontsize=8, facecolor='#1a1a26', edgecolor='#2a2a3e', labelcolor='#e8e8f0')
axes[1,0].spines['top'].set_visible(False); axes[1,0].spines['right'].set_visible(False)

# Top symptoms
axes[1,1].set_facecolor('#12121a')
all_syms = pd.concat([df['symptom_1'],df['symptom_2'],df['symptom_3']]).value_counts().head(10)
axes[1,1].bar(all_syms.index, all_syms.values, color='#7c4dff', edgecolor='#0a0a0f')
axes[1,1].set_title('Most Common Symptoms', color='#e8e8f0', fontsize=11)
axes[1,1].tick_params(axis='x', rotation=45, colors='#8888aa')
axes[1,1].tick_params(axis='y', colors='#8888aa')
axes[1,1].spines['top'].set_visible(False); axes[1,1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 4. NLP Pipeline (spaCy)

> In the production chatbot, user text input is processed through a spaCy NLP pipeline before classification.

In [ ]:
# Simulated spaCy NLP pipeline
# In production: pip install spacy && python -m spacy download en_core_web_sm

def simulate_nlp_pipeline(text):
    """
    Simulates what spaCy NLP pipeline does:
    1. Tokenization
    2. Stop word removal  
    3. Lemmatization
    4. Named entity extraction (symptoms)
    """
    stop_words = {'i','am','have','been','the','a','an','and','or','but',
                  'is','are','was','were','be','do','does','did','my','me'}
    tokens = text.lower().split()
    cleaned = [t for t in tokens if t not in stop_words]
    
    symptom_keywords = [
        'fever','cough','headache','pain','fatigue','nausea','breathing',
        'rash','dizziness','throat','chest','back','vomiting','shortness'
    ]
    extracted_symptoms = [t for t in cleaned if any(k in t for k in symptom_keywords)]
    
    return {
        'original': text,
        'tokens': cleaned,
        'extracted_symptoms': extracted_symptoms
    }

# Test the pipeline
test_inputs = [
    'I have a fever and chills with muscle ache',
    'Chest pain and shortness of breath since morning',
    'My head is throbbing and I feel nauseous'
]

for text in test_inputs:
    result = simulate_nlp_pipeline(text)
    print(f'Input: "{result["original"]}"')
    print(f'Tokens: {result["tokens"]}')
    print(f'Symptoms extracted: {result["extracted_symptoms"]}')
    print()

## 5. Feature Engineering & Preprocessing

In [ ]:
# Encode all categorical features
les = {}
for col in ['symptom_1','symptom_2','symptom_3','severity','age_group']:
    les[col] = LabelEncoder()
    df[col+'_enc'] = les[col].fit_transform(df[col])

le_cond = LabelEncoder()
df['condition_enc'] = le_cond.fit_transform(df['condition'])

le_triage = LabelEncoder()
df['triage_enc'] = le_triage.fit_transform(df['triage_level'])

feat_cols = ['symptom_1_enc','symptom_2_enc','symptom_3_enc',
             'duration_days','severity_enc','age_group_enc']

X = df[feat_cols].values
y = df['condition_enc'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')
print(f'Number of conditions: {len(le_cond.classes_)}')
print(f'Conditions: {list(le_cond.classes_)}')

## 6. Model Training — SVM

In [ ]:
# Support Vector Machine
svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)
svm_acc = accuracy_score(y_test, svm_pred)

cv_svm = cross_val_score(svm, X, y, cv=3, scoring='accuracy')
print('=== SVM Results ===')
print(f'Test Accuracy: {svm_acc:.2%}')
print(f'Cross-Val Accuracy: {cv_svm.mean():.2%} ± {cv_svm.std():.2%}')

## 7. Model Training — Decision Tree

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(max_depth=6, min_samples_split=2, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)

cv_dt = cross_val_score(dt, X, y, cv=3, scoring='accuracy')
print('=== Decision Tree Results ===')
print(f'Test Accuracy: {dt_acc:.2%}')
print(f'Cross-Val Accuracy: {cv_dt.mean():.2%} ± {cv_dt.std():.2%}')
print(f'Tree Depth: {dt.get_depth()}')
print(f'Number of Leaves: {dt.get_n_leaves()}')

## 8. Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0a0a0f')

# Accuracy comparison
axes[0].set_facecolor('#12121a')
models = ['SVM (RBF)', 'Decision Tree']
accs = [svm_acc*100, dt_acc*100]
bars = axes[0].bar(models, accs, color=['#00e5ff','#7c4dff'], edgecolor='#0a0a0f', width=0.5)
for bar, val in zip(bars, accs):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{val:.1f}%', ha='center', color='white', fontsize=12,
                 fontweight='bold', fontfamily='monospace')
axes[0].set_title('Model Accuracy Comparison', color='#e8e8f0', fontsize=11)
axes[0].set_ylabel('Accuracy (%)', color='#8888aa')
axes[0].set_ylim(0,110)
axes[0].tick_params(colors='#8888aa')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Feature importance (DT)
axes[1].set_facecolor('#12121a')
feat_labels = ['Symptom 1','Symptom 2','Symptom 3','Duration','Severity','Age Group']
feat_imp = pd.Series(dt.feature_importances_, index=feat_labels).sort_values()
bars2 = axes[1].barh(feat_imp.index, feat_imp.values,
                     color=['#00e5ff' if v==feat_imp.max() else '#7c4dff' for v in feat_imp.values],
                     edgecolor='#0a0a0f')
axes[1].set_title('Decision Tree Feature Importance', color='#e8e8f0', fontsize=11)
axes[1].set_xlabel('Importance', color='#8888aa')
axes[1].tick_params(colors='#8888aa', labelsize=9)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 9. Chatbot Prediction Function

In [ ]:
def predict_condition(symptom_1, symptom_2, symptom_3, duration, severity, age_group, model='svm'):
    """
    Predict condition and triage level from symptoms.
    In production, symptoms come from spaCy NLP pipeline parsing user input.
    """
    # Encode inputs (handle unseen labels)
    def safe_encode(le, val):
        if val in le.classes_:
            return le.transform([val])[0]
        return 0

    s1 = safe_encode(les['symptom_1'], symptom_1)
    s2 = safe_encode(les['symptom_2'], symptom_2)
    s3 = safe_encode(les['symptom_3'], symptom_3)
    sev = safe_encode(les['severity'], severity)
    age = safe_encode(les['age_group'], age_group)
    
    X_input = np.array([[s1, s2, s3, duration, sev, age]])
    
    clf = svm if model == 'svm' else dt
    pred_enc = clf.predict(X_input)[0]
    condition = le_cond.inverse_transform([pred_enc])[0]
    
    # Map condition to triage
    emergency = ['Heart Attack','Appendicitis','COPD','Asthma']
    urgent = ['Influenza','Angina','Rheumatoid Arthritis','Sciatica','Tonsillitis']
    triage = 'Emergency' if condition in emergency else 'Urgent' if condition in urgent else 'Non-Urgent'
    
    return {'condition': condition, 'triage': triage}

# Test predictions
test_cases = [
    ('fever', 'chills', 'muscle ache', 3, 'moderate', 'adult'),
    ('chest pain', 'sweating', 'jaw pain', 1, 'severe', 'elderly'),
    ('headache', 'nausea', 'light sensitivity', 2, 'moderate', 'adult'),
    ('cough', 'runny nose', 'sneezing', 5, 'mild', 'child'),
    ('shortness of breath', 'wheezing', 'chest tightness', 2, 'severe', 'adult'),
]

print('=== Chatbot Predictions ===')
print(f'{"Symptoms":<45} {"Condition":<25} {"Triage"}')
print('-'*85)
for s1,s2,s3,dur,sev,age in test_cases:
    result = predict_condition(s1,s2,s3,dur,sev,age)
    print(f'{s1+", "+s2:<45} {result["condition"]:<25} {result["triage"]}')

## 10. Voice Input Integration (Production)

> In production, the chatbot accepts voice input via speech-to-text API, which is then fed into the NLP pipeline.

In [ ]:
# Voice input simulation
# In production uses: SpeechRecognition library or Google Speech-to-Text API

def simulate_voice_input(audio_text):
    """
    Simulates voice-to-text conversion.
    Production: import speech_recognition as sr
    """
    print(f'🎤 Voice Input Received: "{audio_text}"')
    nlp_result = simulate_nlp_pipeline(audio_text)
    print(f'📝 Transcribed & Processed: {nlp_result["tokens"]}')
    print(f'🔍 Symptoms Detected: {nlp_result["extracted_symptoms"]}')
    return nlp_result

# Demo
voice_samples = [
    'I have chest pain and I am sweating a lot',
    'Been having a fever with chills for three days',
    'Severe headache with sensitivity to light'
]

print('=== Voice Input Pipeline Demo ===')
for sample in voice_samples:
    print()
    result = simulate_voice_input(sample)

## 11. Summary

| Component | Details |
|-----------|----------|
| NLP Pipeline | spaCy tokenization, lemmatization, symptom extraction |
| Model 1 | SVM with RBF kernel |
| Model 2 | Decision Tree (max_depth=6) |
| Input | Text + Voice (Speech-to-Text API) |
| Output | Condition prediction + Triage level |
| Platform | Mobile + Web via dynamic routing |

### Key Findings
- **Primary symptom** is the most important feature for condition classification
- **SVM** handles multi-class classification well with clear symptom boundaries
- **Decision Tree** provides interpretable rules useful for medical triage logic
- **Voice integration** dramatically improves accessibility for elderly and mobile users